# SBER AI Journey — 3D Mesh Quality Control · полный каталог моделей на диске

Этот ноутбук ничего не обучает — он только читает то, что уже где-то сохранено.

Предыдущая версия каталога, `sber_meshqc_model_zoo.ipynb`, знала только про две папки на Google
Диске (`sber_meshqc/artifacts` для решения 1 и `meshqc_sol2/artifacts` для решения 2). С тех пор
появились ещё как минимум `sber_meshqc_v5/artifacts` (эта тетрадь), собственная папка
`v9_lean`, и вдобавок часть прогонов (например, `dino448_s2`) вручную выгружена бэкапом на
локальный диск (`D:\downloads\dino448_s2_fold*_preds.joblib`). Хардкодить список папок — тупик:
рано или поздно какой-то прогон, где бы он ни лежал, останется незамеченным ансамблем.

Поэтому здесь **нет списка папок вообще** — ноутбук рекурсивно обходит:

- все локальные диски (`C:\`, `D:\`, … на Windows; `/` на Linux/mac — актуально, если ноутбук
  выгружен в Colab и работает с примонтированным Диском как с обычной ФС);
- Google Диск (`/content/drive/MyDrive`), если ноутбук исполняется в Google Colab и Диск
  подключается — это делается автоматически, локальный запуск просто пропустит этот шаг;

и в обоих случаях собирает оба известных формата артефактов, где бы они ни лежали:

1. **Найти на диске все прогоны, которые вообще были обучены**, и какие из них годны
   для ансамбля (раздел 1);
2. **Перебрать пары и тройки моделей с обязательным участием `dino448_s2`** и показать скор
   каждой комбинации (раздел 2). `dino448_s2` — лучшая одиночная модель на момент, когда
   создавался каталог (см. `solution_log_sber_meshqc.md`, этап 6–7), поэтому она взята
   якорем: каждая пара/тройка обязана её включать.

Оценка везде — **вложенная кросс-валидация** (правило порогов настраивается на 4 фолдах,
проверяется на 5-м), а не OOF: OOF ансамбля продолжает расти даже когда честная оценка стоит
на месте — это переобучение решающего правила, а не самой модели.

## 0. Установка зависимостей

In [ ]:
!pip -q install kagglehub iterative-stratification pyarrow
!pip -q install scikit-learn pandas matplotlib joblib

## 1. Все обученные модели на диске

### 1.1. Импорты, конфиг, точки обхода

Оба решения хранят прогоны в разных форматах артефактов, но **где именно лежит файл — не
важно**: ниже мы обходим диск целиком и ищем сами файлы по имени:

| формат артефакта | кто так сохраняет | пример имени |
|---|---|---|
| по фолдам: `{tag}_fold{k}_preds.joblib`, k=0..4 | `sber_meshqc_multiview.ipynb`, `sber_meshqc_multiview_v5.ipynb`, `sber_meshqc_v9_lean.ipynb` | `dinos224_geo2_v5_fold3_preds.joblib` |
| одним файлом: `{tag}_cnn_preds.joblib` | `sber_meshqc_solution2.ipynb`, некоторые ветки v9_lean | `dino448_s2_cnn_preds.joblib` |

`ROOTS` собирается из всех обнаруженных локальных дисков (`C:\`, `D:\`, …) плюс Google Диск,
если он доступен (Colab). `EXCLUDE_DIR_NAMES` отсекает по имени папки заведомо нерелевантные и
дорогие поддеревья (система, кэши пакетов, `node_modules`, `.git` и т.п.) — обход диска на
десятки/сотни гигабайт мелких файлов иначе рискует идти очень долго. `SCAN_TIME_BUDGET_S` —
аварийный выход на случай захламлённого диска: обход останавливается, а не висит бесконечно.

In [ ]:
import os, re, json, math, time, random, string, platform, warnings, itertools, fnmatch
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')

mpl.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#fbfbfd', 'axes.grid': True,
    'grid.alpha': .25, 'grid.linestyle': '--', 'axes.spines.top': False,
    'axes.spines.right': False, 'font.size': 10, 'axes.titlesize': 11,
    'axes.titleweight': 'bold'})
PAL = {'good': '#2e9e5b', 'bad': '#d1495b', 'warn': '#e8a33d', 'main': '#2d6cdf',
       'alt': '#8a4fbd', 'grey': '#9aa0a6', 'dark': '#1f2933'}


class Reporter:
    W = 86
    @classmethod
    def section(cls, t, s=''):
        print('\n' + '=' * cls.W); print(f'  {t}')
        if s: print(f'  {s}')
        print('=' * cls.W)
    @staticmethod
    def kv(**kw):
        for k, v in kw.items(): print(f'    {k:<34s} {v}')
    @staticmethod
    def ok(m): print(f'  [+] {m}')
    @staticmethod
    def warn(m): print(f'  [!] {m}')


R = Reporter

SEED = 42
N_FOLDS = 5
ARTIFACT_COLS = ['abstract', 'artifacts', 'intersection', 'lowpoly', 'noisy',
                 'open', 'partial', 'scale', 'set', 'simple']
TARGET_COLS = ARTIFACT_COLS + ['quality']
CC = ['cnn_' + c for c in TARGET_COLS]
DATASET = 'daniilantonov5/3d-mesh-quality-control'
ANCHOR = 'dino448_s2'          # обязательный якорь ансамбля (раздел 2)
NAN_TOLERANCE = 0.01           # допуск NaN в предсказаниях прогона, см. этап 6 (dino448 BatchNorm)

random.seed(SEED); np.random.seed(SEED)


def local_drive_roots():
    """Локальные диски: C:, D:, ... на Windows; корень файловой системы на Linux/mac."""
    if platform.system() == 'Windows':
        return [Path(f'{d}:/') for d in string.ascii_uppercase if Path(f'{d}:/').exists()]
    return [Path('/')]


# Google Диск — бонус, если ноутбук всё же исполняется в Colab; на обычном локальном запуске
# (как сейчас) этот блок просто промолчит и ничего не подключит.
COLAB_DRIVE = Path('/content/drive/MyDrive')
try:
    from google.colab import drive as _gd
    _gd.mount('/content/drive')
except Exception as e:
    print('Google Colab/Drive недоступны — обычный локальный запуск:', e)

EXTRA_ROOTS = [
    # точечно добавить папку, если полный обход всех локальных дисков слишком долгий, например:
    # Path(r'D:\downloads'),
]

ROOTS = [r for r in ([COLAB_DRIVE] + local_drive_roots() + EXTRA_ROOTS) if r.exists()]

# Папки, обход которых заведомо бессмысленный и дорогой: система, кэши пакетов, чужой код и
# т.п. Отсекаем ПО ИМЕНИ на любом уровне вложенности — тетради решения туда артефакты не пишут.
# Не нашли ожидаемую модель — в первую очередь проверьте, не попала ли её папка сюда.
EXCLUDE_DIR_NAMES = {
    '.git', '.ipynb_checkpoints', '__pycache__', 'node_modules', '.venv', 'venv',
    'site-packages', '.cache', '.conda', '.anaconda',
    'Windows', 'Program Files', 'Program Files (x86)', 'ProgramData',
    '$Recycle.Bin', 'System Volume Information', 'AppData',
}

# Предохранитель: если диск огромный и захламлён, полный обход не должен виснуть навечно.
SCAN_TIME_BUDGET_S = 1800

R.section('Точки обхода', f'{[str(r) for r in ROOTS]}')

### 1.2. Метрика и вспомогательные функции

Дословно скопированы из `sber_meshqc_solution2.ipynb` / `sber_meshqc_model_zoo.ipynb` (метрика
одинаковая во всех тетрадях проекта), чтобы каталог воспроизводился в Colab независимо от того,
какая рабочая тетрадь запускалась в сессии последней.

In [ ]:
def competition_metric(y_true, y_pred):
    """10*F1(quality) + 10*F1_weighted(10 дефектов)."""
    f1_q = f1_score(y_true[:, 10], y_pred[:, 10], zero_division=0)
    f1_a = f1_score(y_true[:, :10], y_pred[:, :10], average='weighted', zero_division=0)
    return 10 * f1_q + 10 * f1_a, f1_q, f1_a


def per_class_f1(y, p):
    return pd.Series([f1_score(y[:, i], p[:, i], zero_division=0) for i in range(11)],
                     index=TARGET_COLS)


def exact_f1_threshold(y, p, plateau=0.995):
    """Точный argmax F1-порога за один проход по отсортированным вероятностям."""
    y = np.asarray(y).astype(np.int32); p = np.asarray(p, np.float64)
    P = int(y.sum())
    if P == 0:
        return 0.5, 0.0
    o = np.argsort(-p, kind='stable'); ys, ps = y[o], p[o]
    f1 = 2 * np.cumsum(ys) / (np.arange(1, len(y) + 1) + P)
    fb = float(f1.max())
    cut = np.empty(len(ps), bool); cut[-1] = True; cut[:-1] = ps[:-1] > ps[1:]
    ok = np.where((f1 >= plateau * fb) & cut)[0]
    if not len(ok):
        ok = np.where(cut)[0][[int(np.argmax(f1[cut]))]]
    i = int(ok[len(ok) // 2])
    return float((ps[i] + ps[i + 1]) / 2 if i + 1 < len(ps) else ps[i] - 1e-6), fb


def quick_thresholds(y, prob):
    return np.array([exact_f1_threshold(y[:, i], prob[:, i])[0] for i in range(11)])

### 1.3. Данные и фолды

Фолды пересобираются тем же `MultilabelStratifiedKFold(seed=42, n_splits=5)`, что и во всех
тетрадях решения, по тем же `target_cols` в том же порядке — при одинаковом датасете и порядке
строк это даёт побитово тот же столбец `fold`, что и во время обучения. Без этого вложенная
оценка сравнивала бы модели по чужим разбиениям.

In [ ]:
import kagglehub
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

root = Path(kagglehub.dataset_download(DATASET))

def _pick_csv(mode):
    cands = list(root.rglob('*.csv'))
    def score(p):
        n = p.name.lower(); s = 0
        if mode == 'train':
            s += 10 * ('train' in n) - 20 * ('submission' in n) - 20 * ('test' in n)
        else:
            s += 10 * ('test' in n) - 5 * ('submission' in n) - 20 * ('train' in n)
        return s - 0.001 * len(n)
    return sorted(cands, key=score, reverse=True)[0]

train_df = pd.read_csv(_pick_csv('train'))
test_df = pd.read_csv(_pick_csv('test'))[['item_id']].copy()
train_df['item_id'] = train_df['item_id'].astype(str)
test_df['item_id'] = test_df['item_id'].astype(str)
if 'quality' not in train_df.columns:
    train_df['quality'] = (train_df[ARTIFACT_COLS].sum(1) == 0).astype(int)

mskf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
train_df['fold'] = -1
for k, (_, vi) in enumerate(mskf.split(train_df, train_df[TARGET_COLS].values)):
    train_df.loc[train_df.index[vi], 'fold'] = k

R.section('Данные', f'train {train_df.shape}, test {test_df.shape}')
print(train_df.groupby('fold')[TARGET_COLS].mean().round(3).to_string())

### 1.4. Полный обход диска

Вместо списка папок — рекурсивный обход каждого корня из `ROOTS` через `os.walk`, с отсечением
`EXCLUDE_DIR_NAMES` на лету (а не постфактум: `os.walk` позволяет вычеркнуть подпапку из
`dirnames`, и спуск в неё вообще не произойдёт — в отличие от `Path.rglob`, который так не
умеет и обошёл бы её всю). На каждом шаге собираются имена файлов, подходящие под оба известных
шаблона артефактов; `SCAN_TIME_BUDGET_S` — аварийный выход, если диск огромный.

Тег ищется где угодно на диске, независимо от того, объявлен он в текущей сессии в
`BRANCHES`/`CONFIGS`/`ensemble_tags` какой-либо тетради или нет — так находятся и модели,
для которых конфиг-ветка уже была удалена или переименована.

In [ ]:
def _iter_matching_files(roots, patterns, exclude_dirnames=EXCLUDE_DIR_NAMES,
                         time_budget=SCAN_TIME_BUDGET_S):
    t0 = time.time()
    n_dirs = n_hits = 0
    for root in roots:
        for dirpath, dirnames, filenames in os.walk(root):
            dirnames[:] = [d for d in dirnames if d not in exclude_dirnames]
            n_dirs += 1
            for fn in filenames:
                if any(fnmatch.fnmatch(fn, pat) for pat in patterns):
                    n_hits += 1
                    yield Path(dirpath) / fn
            if time_budget and (time.time() - t0) > time_budget:
                R.warn(f'обход прерван по лимиту времени ({time_budget:.0f} c) в {dirpath}; '
                       f'часть диска не осмотрена — сузьте ROOTS или увеличьте SCAN_TIME_BUDGET_S')
                print(f'  папок осмотрено: {n_dirs}, файлов-кандидатов: {n_hits}, '
                      f'время: {time.time() - t0:.0f} c')
                return
    print(f'  папок осмотрено: {n_dirs}, файлов-кандидатов: {n_hits}, '
          f'время: {time.time() - t0:.0f} c')


def discover_solution1(roots, n_folds=N_FOLDS):
    """Формат по фолдам: {tag}_fold{k}_preds.joblib, k=0..n_folds-1, где угодно на диске."""
    pattern = re.compile(r'^(.+)_fold(\d+)_preds\.joblib$')
    groups = {}
    for p in _iter_matching_files(roots, ['*_fold*_preds.joblib']):
        m = pattern.match(p.name)
        if not m:
            continue
        tag, k = m.group(1), int(m.group(2))
        groups.setdefault(tag, {}).setdefault(k, p)   # первый найденный побеждает

    models = {}
    for tag, files in sorted(groups.items()):
        have = sorted(files)
        if have != list(range(n_folds)):
            models[tag] = {'error': f'неполный набор фолдов: есть {have}, нужно 0..{n_folds - 1}',
                            'source': 'по фолдам', 'path': str(next(iter(files.values())).parent)}
            continue
        oof = pd.DataFrame({'item_id': train_df['item_id']})
        for c in TARGET_COLS:
            oof['cnn_' + c] = np.nan
        te_parts, broken = [], None
        for k in range(n_folds):
            try:
                d = joblib.load(files[k])
            except Exception as e:
                broken = f'фолд {k} не читается: {e}'; break
            idx = oof.set_index('item_id').index.get_indexer(d['oof_ids'])
            oof.loc[idx, CC] = d['oof']
            te_parts.append(pd.DataFrame(d['te'], columns=CC).assign(item_id=d['te_ids']))
        if broken:
            models[tag] = {'error': broken, 'source': 'по фолдам', 'path': str(files[0].parent)}
            continue
        te = pd.concat(te_parts).groupby('item_id', as_index=False).mean()
        models[tag] = {
            'oof': oof, 'te': te, 'mask': oof['cnn_quality'].notna().values,
            'stored_score': None, 'source': 'по фолдам', 'path': str(files[0].parent),
        }
    return models


def discover_solution2(roots):
    """Формат единым файлом: {tag}_cnn_preds.joblib, где угодно на диске."""
    models = {}
    for p in _iter_matching_files(roots, ['*_cnn_preds.joblib']):
        tag = p.name[:-len('_cnn_preds.joblib')]
        if tag in models:
            continue
        try:
            d = joblib.load(p)
        except Exception as e:
            models[tag] = {'error': f'joblib не читается: {e}', 'source': 'единый файл',
                            'path': str(p.parent)}
            continue
        models[tag] = {
            'oof': d.get('oof'), 'te': d.get('te'), 'mask': d.get('mask'),
            'stored_score': d.get('score'), 'source': 'единый файл', 'path': str(p.parent),
        }
    return models


R.section('Обход по фолдам ({tag}_fold{k}_preds.joblib)')
RAW_MODELS = discover_solution1(ROOTS)
R.section('Обход единых файлов ({tag}_cnn_preds.joblib)')
RAW_MODELS.update(discover_solution2(ROOTS))   # единый файл приоритетнее при совпадении тега

R.section('Найдено на диске', f'{len(RAW_MODELS)} тег(ов)')
for tag, rec in sorted(RAW_MODELS.items()):
    tail = rec['error'] if 'error' in rec else 'OK'
    print(f'  {tag:<20s} [{rec["source"]:<10s}] {rec.get("path", "?"):<55s} {tail}')

### 1.5. Валидация

Критерии — из собственного опыта проекта (этапы 5–6 `solution_log_sber_meshqc.md`):

- в артефакте есть все 11 столбцов `cnn_*` и на train, и на test;
- есть хотя бы один объект с OOF-предсказанием, и покрыто не меньше половины train;
- доля `NaN` в предсказаниях (OOF и test) не больше **1%** — тот же допуск, который проект
  ввёл после сбоя `dino448` (2 из 5 фолдов уходили в `NaN` от переполнения BatchNorm, а строгое
  требование «`NaN` вообще нет» тогда выбросило из ансамбля все рабочие модели);
- вероятности не выходят за пределы `[0, 1]` (после санации `NaN`→0.5, `inf`→клип) больше, чем
  у 1% значений — признак того же переполнения статистик, что было у `dino448`;
- ни один из 11 классов не предсказывается константой (нулевая дисперсия = модель ничего не
  выучила либо веса не загрузились).

Скор каждой модели **пересчитывается заново** по единой процедуре (`quick_thresholds` +
`competition_metric` на общих фолдах), а не берётся из сохранённого `score` — так все модели
сравнимы между собой независимо от того, какая тетрадь и в какой момент их считала.

In [ ]:
def validate_model(tag, rec):
    if 'error' in rec:
        return False, rec['error'], {}

    oof, te, mask = rec['oof'], rec['te'], rec['mask']
    missing = [c for c in CC if c not in oof.columns or c not in te.columns]
    if missing:
        return False, f'нет колонок {missing}', {}

    mask = np.asarray(mask, bool)
    cov = int(mask.sum())
    if cov == 0:
        return False, 'нет ни одного объекта с OOF-предсказанием', {}

    P = oof.loc[mask, CC].values.astype(np.float64)
    nan_oof = float(np.isnan(P).mean())
    Pte_raw = te.set_index('item_id').reindex(test_df['item_id'])[CC].values.astype(np.float64)
    te_cov = float(np.isfinite(Pte_raw).all(1).mean())
    nan_te = float(np.isnan(Pte_raw).mean())
    finite_vals = P[np.isfinite(P)]
    range_bad = float(((P < -1e-6) | (P > 1 + 1e-6))[np.isfinite(P)].mean()) if finite_vals.size else 1.0

    reasons = []
    if cov < len(train_df) * 0.5:
        reasons.append(f'покрытие OOF всего {cov}/{len(train_df)}')
    if nan_oof > NAN_TOLERANCE:
        reasons.append(f'NaN в OOF {nan_oof:.1%} > {NAN_TOLERANCE:.0%}')
    if nan_te > NAN_TOLERANCE:
        reasons.append(f'NaN в test {nan_te:.1%} > {NAN_TOLERANCE:.0%}')
    if te_cov < 1 - NAN_TOLERANCE:
        reasons.append(f'test покрыт лишь на {te_cov:.1%} по item_id')
    if range_bad > NAN_TOLERANCE:
        reasons.append(f'вероятности вне [0,1] у {range_bad:.1%} значений (похоже на взрыв BatchNorm)')
    const_cols = [TARGET_COLS[i] for i in range(11) if np.nanstd(P[:, i]) < 1e-9]
    if const_cols:
        reasons.append(f'вырожденные (постоянные) предсказания: {const_cols}')

    Psan = np.clip(np.nan_to_num(P, nan=0.5, posinf=1.0, neginf=0.0), 0, 1)
    y = train_df.loc[mask, TARGET_COLS].values
    th = quick_thresholds(y, Psan)
    score = competition_metric(y, (Psan > th).astype(int))[0]

    info = {'coverage': cov, 'test_coverage': te_cov, 'nan_oof': nan_oof, 'nan_te': nan_te,
            'range_bad': range_bad, 'recomputed_score': score,
            'stored_score': rec.get('stored_score')}
    return (len(reasons) == 0), ('; '.join(reasons) if reasons else 'ок'), info


VALID_MODELS, rows = {}, []
for tag, rec in RAW_MODELS.items():
    ok, reason, info = validate_model(tag, rec)
    row = {'модель': tag, 'формат': rec.get('source', '?'), 'папка': rec.get('path', '?'),
           'покрытие train': info.get('coverage', 0),
           'покрытие test, %': round(100 * info.get('test_coverage', 0), 1),
           'NaN OOF, %': round(100 * info.get('nan_oof', np.nan), 2) if info else np.nan,
           'скор (пересчитан)': round(info.get('recomputed_score', np.nan), 3) if info else np.nan,
           'скор (сохранён)': (round(info['stored_score'], 3)
                               if info.get('stored_score') is not None else '—'),
           'валиден': 'да' if ok else 'нет', 'причина': reason}
    rows.append(row)
    if ok:
        P_full = np.full((len(train_df), 11), np.nan)
        mask = np.asarray(rec['mask'], bool)
        P_full[mask] = np.clip(np.nan_to_num(
            rec['oof'].loc[mask, CC].values.astype(np.float64), nan=0.5, posinf=1.0, neginf=0.0), 0, 1)
        Pte_full = np.clip(np.nan_to_num(
            rec['te'].set_index('item_id').reindex(test_df['item_id'])[CC].values.astype(np.float64),
            nan=0.5, posinf=1.0, neginf=0.0), 0, 1)
        VALID_MODELS[tag] = {'mask': mask, 'P_oof_full': P_full, 'P_te_full': Pte_full,
                              'source': rec.get('source'), 'path': rec.get('path'),
                              'score': info['recomputed_score']}

summary = pd.DataFrame(rows).sort_values('валиден', ascending=False).sort_values(
    'скор (пересчитан)', ascending=False, na_position='last')
R.section('Сводка по всем найденным на диске моделям')
print(summary.to_string(index=False))
R.kv(**{'валидных моделей': len(VALID_MODELS), 'всего найдено': len(RAW_MODELS)})

In [ ]:
valid_summary = summary[summary['валиден'] == 'да'].sort_values('скор (пересчитан)', ascending=False)
fig, ax = plt.subplots(figsize=(9, 0.45 * max(len(valid_summary), 1) + 1.5))
colors = [PAL['main'] if f == 'единый файл' else PAL['alt'] for f in valid_summary['формат']]
ax.barh(valid_summary['модель'], valid_summary['скор (пересчитан)'], color=colors)
for y, v in enumerate(valid_summary['скор (пересчитан)']):
    ax.text(v + 0.05, y, f'{v:.2f}', va='center', fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('пересчитанный OOF-скор'); ax.set_title('Валидные модели, найденные на диске')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=PAL['main'], label='единый файл ({tag}_cnn_preds.joblib)'),
                   Patch(color=PAL['alt'], label='по фолдам ({tag}_fold{k}_preds.joblib)')],
          loc='lower right', fontsize=9)
plt.tight_layout(); plt.show()

invalid = summary[summary['валиден'] == 'нет']
if len(invalid):
    R.section('Отбракованные модели')
    for _, r in invalid.iterrows():
        R.warn(f"{r['модель']} [{r['папка']}]: {r['причина']}")
else:
    R.ok('Отбракованных моделей нет')

## 2. Перебор пар и троек с `dino448_s2`

`dino448_s2` — обязательный участник каждой комбинации (якорь), найденный где угодно на диске в
разделе 1. Перебираются:

- **пары** — `dino448_s2` + одна любая другая валидная модель;
- **тройки** — `dino448_s2` + любые две другие валидные модели.

Веса в бленде — равные (как `score_eq` в §11 `sber_meshqc_solution2.ipynb`): по-классовые и
даже общие веса, подобранные по OOF, в проекте не раз переобучались (этапы 2 и 7 лога), поэтому
для честного сравнения комбинаций между собой сначала берётся простое среднее, а
скор-пропорциональные веса — только как отдельная проверка лучших кандидатов в §2.5.

In [ ]:
if ANCHOR not in VALID_MODELS:
    raise RuntimeError(
        f'Якорь {ANCHOR!r} не найден среди валидных моделей нигде на диске. '
        f'Проверьте, что {ANCHOR}_cnn_preds.joblib (или полный набор '
        f'{ANCHOR}_fold*_preds.joblib) действительно лежит на диске, '
        f'и что ROOTS/EXCLUDE_DIR_NAMES в §1.1 не отсекли нужную папку.')

OTHERS = sorted(t for t in VALID_MODELS if t != ANCHOR)
R.section('Участники перебора', f'якорь: {ANCHOR} | остальные валидные модели: {OTHERS}')


def blend_models(tags, weights=None):
    weights = weights or {t: 1.0 for t in tags}
    ws = sum(weights[t] for t in tags)
    mask = np.ones(len(train_df), bool)
    for t in tags:
        mask &= VALID_MODELS[t]['mask']
    Po = np.zeros((int(mask.sum()), 11))
    Pt = np.zeros((len(test_df), 11))
    for t in tags:
        w = weights[t] / ws
        Po += w * VALID_MODELS[t]['P_oof_full'][mask]
        Pt += w * VALID_MODELS[t]['P_te_full']
    return mask, Po, Pt


def eval_combo(tags, weights=None):
    mask, Po, _ = blend_models(tags, weights)
    y = train_df.loc[mask, TARGET_COLS].values
    th = quick_thresholds(y, Po)
    oof_score = competition_metric(y, (Po > th).astype(int))[0]

    folds_masked = train_df.loc[mask, 'fold'].values
    nested = []
    for k in range(N_FOLDS):
        tr_i, va_i = folds_masked != k, folds_masked == k
        if va_i.sum() == 0:
            continue
        th_k = quick_thresholds(y[tr_i], Po[tr_i])
        nested.append(competition_metric(y[va_i], (Po[va_i] > th_k).astype(int))[0])
    nested = np.array(nested)
    return {'coverage': int(mask.sum()), 'oof_score': oof_score,
            'nested_mean': float(nested.mean()) if len(nested) else np.nan,
            'nested_std': float(nested.std()) if len(nested) else np.nan,
            'n_folds_used': len(nested)}, mask


def mean_error_corr(tags, mask):
    if len(tags) < 2:
        return np.nan
    y = train_df.loc[mask, TARGET_COLS].values
    errs = {t: np.abs(VALID_MODELS[t]['P_oof_full'][mask] - y).mean(1) for t in tags}
    vals = [np.corrcoef(errs[a], errs[b])[0, 1] for a, b in itertools.combinations(tags, 2)]
    return float(np.mean(vals))


solo_metrics, _ = eval_combo((ANCHOR,))
R.section(f'Базовый уровень: только {ANCHOR}',
          f"OOF {solo_metrics['oof_score']:.3f} | вложенная "
          f"{solo_metrics['nested_mean']:.3f} ± {solo_metrics['nested_std']:.3f}")

In [ ]:
def combo_row(tags):
    metrics, mask = eval_combo(tags)
    metrics['комбинация'] = ' + '.join(tags)
    metrics['участников'] = len(tags)
    metrics['корр. ошибок'] = mean_error_corr(tags, mask)
    metrics['delta_vs_solo'] = metrics['nested_mean'] - solo_metrics['nested_mean']
    return metrics

pair_rows = [combo_row((ANCHOR, t)) for t in OTHERS]
triple_rows = [combo_row((ANCHOR, t1, t2)) for t1, t2 in itertools.combinations(OTHERS, 2)]

cols = ['комбинация', 'участников', 'coverage', 'oof_score', 'nested_mean', 'nested_std',
        'delta_vs_solo', 'корр. ошибок']

pairs_df = pd.DataFrame(pair_rows)[cols].sort_values('nested_mean', ascending=False) if pair_rows \
    else pd.DataFrame(columns=cols)
triples_df = pd.DataFrame(triple_rows)[cols].sort_values('nested_mean', ascending=False) if triple_rows \
    else pd.DataFrame(columns=cols)

R.section('Пары: dino448_s2 + 1 модель', f'{len(pairs_df)} комбинаций')
print(pairs_df.round(3).to_string(index=False))

R.section('Тройки: dino448_s2 + 2 модели', f'{len(triples_df)} комбинаций')
print(triples_df.round(3).head(15).to_string(index=False))
if len(triples_df) > 15:
    print(f'  ... и ещё {len(triples_df) - 15} комбинаций (см. triples_df целиком)')

In [ ]:
top_n = 10
top_pairs = pairs_df.head(top_n)
top_triples = triples_df.head(top_n)
plot_df = pd.concat([top_pairs.assign(группа='пары'), top_triples.assign(группа='тройки')])
plot_df = plot_df.sort_values('nested_mean')

fig, ax = plt.subplots(figsize=(9, 0.4 * len(plot_df) + 1.5))
colors = [PAL['main'] if g == 'пары' else PAL['alt'] for g in plot_df['группа']]
ax.barh(plot_df['комбинация'], plot_df['nested_mean'], xerr=plot_df['nested_std'],
        color=colors, ecolor=PAL['grey'], capsize=2)
ax.axvline(solo_metrics['nested_mean'], color=PAL['dark'], ls='--', lw=1.2,
           label=f'{ANCHOR} один ({solo_metrics["nested_mean"]:.2f})')
ax.set_xlabel('вложенная оценка (среднее по 5 фолдам)')
ax.set_title(f'Топ-{top_n} пар и троек с {ANCHOR} (по всему диску)')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=PAL['main'], label='пара'), Patch(color=PAL['alt'], label='тройка'),
                   plt.Line2D([0], [0], color=PAL['dark'], ls='--', label=f'{ANCHOR} один')],
          loc='lower right', fontsize=8)
plt.tight_layout(); plt.show()

best_pair = pairs_df.iloc[0] if len(pairs_df) else None
best_triple = triples_df.iloc[0] if len(triples_df) else None
R.section('Лучшие найденные комбинации')
if best_pair is not None:
    R.kv(**{'лучшая пара': f"{best_pair['комбинация']} — вложенная {best_pair['nested_mean']:.3f} "
                           f"({best_pair['delta_vs_solo']:+.3f} к одиночному {ANCHOR})"})
if best_triple is not None:
    R.kv(**{'лучшая тройка': f"{best_triple['комбинация']} — вложенная {best_triple['nested_mean']:.3f} "
                             f"({best_triple['delta_vs_solo']:+.3f} к одиночному {ANCHOR})"})

### 2.5. Проверка: помогают ли веса, пропорциональные скору

Быстрая диагностика для лучшей пары и лучшей тройки — не как замена основного (равновесного)
сравнения, а как отдельная проверка. Вес каждого участника берётся пропорционально его
пересчитанному одиночному скору из раздела 1.

In [ ]:
def score_weights(tags):
    w = {t: VALID_MODELS[t]['score'] for t in tags}
    return w

for name, tags in (('лучшая пара', tuple(best_pair['комбинация'].split(' + ')) if best_pair is not None else None),
                    ('лучшая тройка', tuple(best_triple['комбинация'].split(' + ')) if best_triple is not None else None)):
    if tags is None:
        continue
    eq_metrics, _ = eval_combo(tags)
    w_metrics, _ = eval_combo(tags, score_weights(tags))
    R.section(f'{name}: {" + ".join(tags)}')
    R.kv(**{'равные веса, вложенная': f"{eq_metrics['nested_mean']:.3f} ± {eq_metrics['nested_std']:.3f}",
            'веса по скору, вложенная': f"{w_metrics['nested_mean']:.3f} ± {w_metrics['nested_std']:.3f}",
            'разница': f"{w_metrics['nested_mean'] - eq_metrics['nested_mean']:+.3f}"})

### 2.6. Итог

Ноутбук не переписывает `submission.csv` — он только показывает, какая комбинация моделей,
обнаруженных **где угодно на диске**, выглядит сильнее по вложенной оценке. Разница
меньше ~0.1–0.2 между соседними комбинациями не является надёжным доказательством превосходства
(см. этап 2 `solution_log_sber_meshqc.md` про шум лидерборда и переобучение решающего правила) —
итоговый выбор членов ансамбля и их весов стоит переносить в рабочую тетрадь только если
преимущество воспроизводится стабильно.

Раздел 1 (столбец «папка») показывает, где физически лежит каждый найденный прогон — это
единственное, что стоит сверить руками, если какой-то ожидаемый тег не появился в списке
валидных моделей.

## 3. Сабмит и метрики для конкретных троек

Из таблицы `triples_df` (раздел 2) выбраны три лучшие тройки — все на базе
`dino448_s2 + cnxb336`, различаются только третьим участником:

| комбинация | вложенная | разброс | корр. ошибок |
|---|---|---|---|
| `dino448_s2 + cnxb336 + dinos224_geo2_v5` | 14.848 | 0.222 | 0.843 |
| `dino448_s2 + cnxb336 + v5_224` | 14.837 | 0.169 | 0.843 |
| `dino448_s2 + cnxb336 + vitb336` | — (см. пересчёт ниже) | — | — |

Ниже для каждой тройки: блендим OOF/test в logit-пространстве (равные веса — та же
логика, что и в `blend_models`/`eval_combo`), подбираем пороги по OOF, считаем
вложенную оценку, приводим `quality` к тождеству разметки (`quality=1` ⟺ нет ни
одного дефекта — как в §12 `sber_meshqc_ensemble3`) и пишем отдельный
`submission_<теги>.csv` на каждую тройку, чтобы можно было сравнить и отправить
любую из них не пересчитывая остальное.

In [ ]:
CANDIDATE_COMBOS = [
    ('dino448_s2', 'cnxb336', 'dinos224_geo2_v5'),
    ('dino448_s2', 'cnxb336', 'v5_224'),
    ('dino448_s2', 'cnxb336', 'vitb336'),
]


def postprocess_quality(pred):
    """quality приводится к тождеству разметки: нет дефектов <=> quality=1."""
    out = pred.copy()
    out[:, 10] = (out[:, :10].sum(1) == 0).astype(int)
    return out


def combo_slug(tags):
    return '_'.join(tags)


def build_submission(tags):
    missing = [t for t in tags if t not in VALID_MODELS]
    if missing:
        R.warn(f'{" + ".join(tags)}: нет в валидных моделях {missing} — пропуск')
        return None

    mask, Po, Pt = blend_models(tags)
    y = train_df.loc[mask, TARGET_COLS].values
    th = quick_thresholds(y, Po)

    pred_oof = (Po > th).astype(int)
    oof_score, f1_q, f1_a = competition_metric(y, pred_oof)

    folds_masked = train_df.loc[mask, 'fold'].values
    nested = []
    for k in range(N_FOLDS):
        tr_i, va_i = folds_masked != k, folds_masked == k
        if va_i.sum() == 0:
            continue
        th_k = quick_thresholds(y[tr_i], Po[tr_i])
        nested.append(competition_metric(y[va_i], (Po[va_i] > th_k).astype(int))[0])
    nested = np.array(nested)

    pred_te = postprocess_quality((Pt > th).astype(int))

    sub = pd.DataFrame(pred_te, columns=TARGET_COLS)
    sub.insert(0, 'item_id', test_df['item_id'].values)
    assert len(sub) == len(test_df) and sub['item_id'].is_unique

    out_name = f'submission_{combo_slug(tags)}.csv'
    sub.to_csv(out_name, index=False)

    pcf = per_class_f1(y, pred_oof)
    corr = mean_error_corr(tags, mask)

    R.section(' + '.join(tags), out_name)
    R.kv(**{
        'покрытие (train)': f'{int(mask.sum())}/{len(train_df)}',
        'OOF метрика': f'{oof_score:.3f} (quality {10 * f1_q:.2f} + artefacts {10 * f1_a:.2f})',
        'вложенная оценка': (f'{nested.mean():.3f} ± {nested.std():.3f}'
                            if len(nested) else '—'),
        'корр. ошибок между членами': f'{corr:.3f}' if not np.isnan(corr) else '—',
        'quality=1 на тесте': f"{int(sub['quality'].sum())}/{len(sub)}",
    })
    print(pcf.round(3).to_string())

    return {'tags': tags, 'file': out_name, 'oof_score': oof_score,
            'nested_mean': float(nested.mean()) if len(nested) else np.nan,
            'nested_std': float(nested.std()) if len(nested) else np.nan,
            'coverage': int(mask.sum()), 'corr': corr}


RESULTS_COMBOS = [r for r in (build_submission(t) for t in CANDIDATE_COMBOS) if r is not None]

summary_combos = pd.DataFrame([{
    'комбинация': ' + '.join(r['tags']), 'файл': r['file'],
    'OOF': round(r['oof_score'], 3), 'вложенная': round(r['nested_mean'], 3),
    'разброс': round(r['nested_std'], 3), 'корр. ошибок': round(r['corr'], 3),
} for r in RESULTS_COMBOS]).sort_values('вложенная', ascending=False)

R.section('Итоговое сравнение сабмитов')
print(summary_combos.to_string(index=False))